# 第 59 章：Capstone 外部合作边界与客座项目准入

这个 notebook 用一个极小的 guest-intake table，对比“只按合作价值排序”的 naive baseline 和“先检查边界、host owner、scope 与客座材料”的合作准入 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_external_collaboration_guest_intake_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['collaboration_value_score'] = float(row['collaboration_value_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} guest-intake items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Partner type:', ordered_counts(rows, 'partner_type'))
print('Guest material:', ordered_counts(rows, 'guest_material_status'))
print('Boundary status:', ordered_counts(rows, 'boundary_status'))
print('Host owner:', ordered_counts(rows, 'host_owner_status'))
print('Scope status:', ordered_counts(rows, 'scope_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['intake_id']
    for row in rows
    if row['reference_route'] == 'intake_ready'
)
top4_value = sorted(
    rows,
    key=lambda row: row['collaboration_value_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'intake_ready'
    for row in top4_value
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'intake_ready'
    for row in top4_value
)
baseline_ready_precision = baseline_ready_hits / len(top4_value)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_value)

print('Reference intake-ready items:', reference_ready_ids)
print('Top-4 by collaboration value:')
for row in top4_value:
    print(
        f"  {row['intake_id']}: value={row['collaboration_value_score']:.2f}, "
        f"route={row['reference_route']}, area={row['collaboration_area']}"
    )
print('Baseline intake precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_intake_item(row):
    if row['boundary_status'] != 'clear':
        return 'clarify_boundary_before_intake', 'data, authorship, or permission boundary is not clear yet'
    if row['host_owner_status'] != 'assigned':
        return 'assign_host_owner', 'course-side host owner is missing'
    if row['scope_status'] != 'bounded':
        return 'narrow_partner_scope', 'collaboration scope is too broad for this course run'
    if row['guest_material_status'] != 'reviewed':
        return 'review_guest_material_before_intake', 'guest material still needs review before entering the course'
    return 'intake_ready', 'collaboration is bounded, reviewed, owned, and safe to intake'


routed_rows = []
for row in rows:
    route, reason = route_intake_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Guest-intake workflow routes:')
for row in routed_rows:
    print(
        f"  {row['intake_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'intake_ready']
review_queue = [row for row in routed_rows if row['route'] == 'review_guest_material_before_intake']
boundary_queue = [row for row in routed_rows if row['route'] == 'clarify_boundary_before_intake']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_host_owner']
scope_queue = [row for row in routed_rows if row['route'] == 'narrow_partner_scope']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'intake_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Intake-ready queue:', [row['intake_id'] for row in ready_queue])
print('Review-material queue:', [row['intake_id'] for row in review_queue])
print('Clarify-boundary queue:', [row['intake_id'] for row in boundary_queue])
print('Assign-owner queue:', [row['intake_id'] for row in owner_queue])
print('Narrow-scope queue:', [row['intake_id'] for row in scope_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured intake precision:', round(ready_precision, 3))


In [ ]:
def intake_steps(row):
    steps = []
    if row['boundary_status'] != 'clear':
        steps.append('先澄清数据权限、作者归属、公开范围和保密边界。')
    if row['host_owner_status'] != 'assigned':
        steps.append('指定课程内部 host owner，负责对接、解释和兜底。')
    if row['scope_status'] != 'bounded':
        steps.append('收窄合作范围，明确只进入哪些周、哪些学生、哪些任务。')
    if row['guest_material_status'] != 'reviewed':
        steps.append('先审 guest material：slides、数据、案例说明和学生可接触范围。')
    return steps or ['可以进入 guest-project intake package；保留边界说明与内部 host note。']


for row in routed_rows:
    if row['route'] != 'intake_ready':
        print(f"\n{row['intake_id']} -> {row['route']} ({row['collaboration_area']})")
        for step in intake_steps(row):
            print(' -', step)


In [ ]:
intake_outline = [
    'Partner identity: lab, observatory, industry, outreach, alumni, or visiting mentor',
    'Course purpose: guest lecture, mini-project, short consult, data demo, or challenge prompt',
    'Boundary note: permissions, authorship, confidentiality, and student-facing limits',
    'Host owner: who on the course side is responsible for intake and follow-up',
    'Scope note: which weeks, students, and deliverables the collaboration actually touches',
    'Exit condition: how the course pauses, narrows, or drops the collaboration if conditions change',
]

intake_assistant_prompt = '''你是我的 capstone 外部合作准入助手。

任务：
1. 先阅读 guest-intake table，不要只按 collaboration value 排序；
2. 先检查 boundary；
3. 再检查 host owner、scope 和 guest material；
4. 把每个合作 route 到 intake_ready、review_guest_material_before_intake、
   clarify_boundary_before_intake、assign_host_owner 或 narrow_partner_scope；
5. 对每个非 ready 合作输出准入前 checklist。

输出格式：
- Intake-ready collaborations
- Review-material collaborations
- Clarify-boundary collaborations
- Assign-owner collaborations
- Narrow-scope collaborations
- Intake checklist by collaboration
'''

print('Guest-intake outline:')
for item in intake_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(intake_assistant_prompt)


In [ ]:
intake_snapshot = {
    'dataset': DATA_PATH.name,
    'n_intake_items': len(rows),
    'baseline_top4_intake_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'intake_ready': [row['intake_id'] for row in ready_queue],
    'review_guest_material_before_intake': [row['intake_id'] for row in review_queue],
    'clarify_boundary_before_intake': [row['intake_id'] for row in boundary_queue],
    'assign_host_owner': [row['intake_id'] for row in owner_queue],
    'narrow_partner_scope': [row['intake_id'] for row in scope_queue],
    'python_version': platform.python_version(),
}

print('Guest-intake snapshot:')
for key, value in intake_snapshot.items():
    print(f'  {key}: {value}')
